# Example 3: improve a PHOENIX fit one assumption at a time

This notebook teaches the fit-planning workflow. We build a reviewed setup, inspect the proposed windows and masks, then optionally run fit variants in a controlled way.

The expensive cells are off by default.

## What this example teaches

- how to compare PHOENIX setup choices one assumption at a time;
- how region selection, masks, resolution, continuum degree, and search mode affect a result;
- why stronger optimization is a stability check, not proof of physical correctness.

## Requirements

The planning/audit path uses bundled data only. Running the optional fit variants requires a configured local PHOENIX library.

## Expected outputs

Setup summaries always run. If fitting is enabled, the notebook prints progress, result summaries, and a compact comparison table.

Note: The most expensive variant is not automatically the best scientific answer; it only tests whether the solution changes under a stronger search budget.


## 1. Load the same X-SHOOTER UVB spectrum

Using the same spectrum as Example 2 lets us reuse the diagnostic-window and masking ideas you just saw.


In [ ]:
import Spyctres as sp

# Bundled X-SHOOTER UVB example spectrum.
spectrum_path = sp.example_data_path("TOO_Gaia21ccu_SCI_SLIT_FLUX_MERGE1D_UVB.fits")
reader = "xshooter_merge1d"

spec = sp.read_spectrum(spectrum_path, reader=reader)
print(spec.summary())


## 2. Pick diagnostic windows and a reviewed mask

These are the pieces we will feed into the fit setup. The selected regions keep the fit focused, and the mask keeps known bad product regions out of the fit.


In [ ]:
# Choose a compact set of useful diagnostic regions.
windows = sp.select_diagnostic_windows(spec, max_windows=6)
print(windows.summary_text(max_rows=6))

# Build a reviewed mask. Archive/product bad regions are excluded; tellurics
# are shown as warnings because this UVB example is not dominated by them.
reviewed_mask = sp.build_mask(spec, archive="mask", tellurics="warn", dibs=False)
print()
print(reviewed_mask.summary_text())

# Plot only the windows that are likely to matter for the fit.
sp.plot_spectrum_line_windows(
    spec.wave,
    spec.flux,
    windows.selected[:4],
    valid_mask=reviewed_mask.valid_mask,
    title="Example 3: observed windows that will drive the fit",
    ncols=2,
    figsize_per_panel=(7.2, 3.4),
)


There's not much to be seen in the `ch_g_band` so we do not need to try fitting it. This band is mainly useful for cooler FGK stars and this UVB spectrum is quite hot/blue.

## 3. Build setup variants

A `FitSetup` is a reviewed plan. The variants below show a useful habit: change one assumption at a time, then compare the effect. This is much safer than silently changing windows, masks, resolution, continuum, and search budget all at once.

You do not need to inspect every setup in everyday use, but we show the sequence here to make the assumptions visible: first Spyctres chooses a quick default, then we optionally add reviewed wavelength regions, an explicit resolution assumption, and a stronger search mode. In practice, inspect the audit plot and the final setup you plan to run.


In [ ]:
# Convert the selected diagnostic-window records into stable region ids.
region_ids = [item["id"] for item in windows.selected[:3]]

# This is an approximate constant resolving-power assumption for the tutorial.
# For precision work, replace it with validated product/instrument LSF metadata.
assumed_R = 6200.0

# 1. Start from Spyctres' quicklook recommendation.
quick_setup = sp.suggest_fit_setup(
    spec,
    mode="quicklook",
    intent="quicklook_classification",
)

# Add assumptions one at a time.
# 2. This is useful when you have manually changed the wavelength windows being fitted
region_setup = quick_setup.with_regions(region_ids)

# 3. This is useful when you have added an explicit resolution assumption
resolution_setup = region_setup.with_resolution(R=assumed_R)

# 4. This is useful when you have altered or want to specify continuum flexibility
continuum_setup = resolution_setup.with_continuum_degree(1)

# 5. This is the final search condfiguration
standard_setup = sp.suggest_fit_setup(
    spec,
    mode="standard",
    intent="quicklook_classification",
    assumed_resolution=assumed_R,
).with_regions(region_ids)

# The above standard_setup call will use whatever continuum setting 
# suggest_fit_setup(..., mode="standard") chooses by default.
# If you want to change it you can do so like this:
# standard_setup = standard_setup.with_continuum_degree(2)

In [ ]:
# Audit view
sp.plot_spectrum(
    spec,
    show_masks=True,
    mask=reviewed_mask,
    diagnostic_selection=windows,
    show_diagnostic_windows=True,
    show_tellurics=True,
    show_nonstellar=True,
    title="Example 3: audit view before fitting",
)

# Print compact summaries so the assumptions are visible before fitting.
# Review them before starting the expensive fitting.
setup_rows = [
    ("quick", quick_setup),
    ("regions", region_setup),
    ("regions + R", resolution_setup),
    ("regions + R + continuum", continuum_setup),
    ("standard search", standard_setup),
]

for label, setup in setup_rows:
    s = setup.summary()
    print(
        f"{label:24s} "
        f"mode={s['mode']:9s} "
        f"window={s.get('recommended_window_label')} "
        f"resolution={s.get('resolution_summary')} "
        f"ready={s.get('ready_for_intent')} "
        f"blockers={', '.join(s.get('blockers_for_intent') or []) or 'none'}"
    )


Some metadata-warning regions shown in purple here lie outside the fitted UVB wavelength range. They are shown because the X-SHOOTER reader carries generic product/telluric warning metadata for broader X-SHOOTER coverage, but they do not affect this UVB-only fit unless they overlap the spectrum and are included in the reviewed valid mask. In this example they are mostly reminders/provenance, not active exclusions.

## 4. Optional PHOENIX fit comparison

Before enabling this cell, ensure you have PHOENIX installed. Run:

`spyctres doctor --require-phoenix`

or, if the console command is not on your path:

`python -m Spyctres.cli doctor --require-phoenix`

We want to explore how different assumptions affect our fits. The comparison table is more important than any one result. If parameters move dramatically between variants, the result is not stable yet.

For ordinary quick use, inspect the audit plot, choose one reviewed setup, and run only that setup.


Note that PHOENIX fitting can take from seconds to minutes depending on whether the template cache is already warm and how many local optimizer starts are requested. Spyctres emits structured progress events during long operations; the fitting cell below can print their elapsed time and milestone messages.

In [ ]:
# Set this False if you want a quieter notebook cell.
SHOW_PROGRESS = True

In [ ]:
# Leave this False for a quick tutorial pass. Set to True to do the actual fitting.
RUN_VARIANT_FITS = True

# Try out fitting under different assumptions
if RUN_VARIANT_FITS:
    variant_results = []
    variant_labels = []
    for label, setup, valid_mask in [
        ("quick", quick_setup, None),
        ("regions + mask", region_setup, reviewed_mask.valid_mask),
        ("regions + R + continuum", continuum_setup, reviewed_mask.valid_mask),
        ("standard search", standard_setup, reviewed_mask.valid_mask),
    ]:
        # If readiness blocks interpretation, we can still run an exploratory
        # tutorial fit, but the reason is recorded in the setup provenance.
        reviewed_setup = setup
        if setup.summary()["ready_for_intent"] is not True:
            reviewed_setup = setup.allow_exploratory(
                reason="Example 3 tutorial comparison; not a final science result."
            )
        # fit_stellar_spectrum() fits all selected windows together by
        # comparing the observed spectrum to a stellar-atmosphere model (PHOENIX)
        result = sp.fit_stellar_spectrum(
            spec,
            model="phoenix",
            setup=reviewed_setup,
            valid_mask=valid_mask,
            progress_callback=(
                lambda event: print(f"[{event.elapsed_s:6.1f}s] {event}", flush=True)
                if SHOW_PROGRESS
                else None
            ),
        )
        print("\n" + label)
        print(result.summary_text(include_hash=False, max_flags=5))
        variant_labels.append(label)
        variant_results.append(result)

    comparison = sp.compare_fits(variant_results, labels=variant_labels)
    print(sp.format_fit_comparison_table(comparison))
    comparison
else:
    print("RUN_VARIANT_FITS is False, so this notebook only reviewed the fit plan.")


In [ ]:
# If the optional fit variants were run, plot the final variant
if RUN_VARIANT_FITS and variant_results:
    sp.plot_model_line_windows(
        variant_results[-1],
        windows=windows.selected[:3],
        segment=spec,
        title="Example 3: final variant in diagnostic windows",
        ncols=2,
        figsize_per_panel=(7.2, 3.7),
    )


### Interpreting the fit variants

These variants were assumption-sensitivity checks. In this run, the reviewed-window fits give similar RV, logg, and metallicity, and the broader `standard search` does not jump to a completely different solution. That is encouraging: the quicklook classification is stable at the broad level.

However, the result is still only exploratory. The reduced χ² values remain high, the spectrum is flagged for artifact review, and the temperature is pushed to the hot edge of the tested grid. Read this as “Spyctres consistently prefers a hot-star solution under these assumptions”. But this is not yet a final precise Teff/logg/[Fe/H] measurement. Despite the flags, the fit was allowed because we explicitly used `.allow_exploratory(...)`, and that decision is recorded in the result. 

Spyctres is telling us that it has found a plausible hot-star quicklook solution, but it is also warning us not to treat the quoted parameters as final yet.

The main lesson here is that reviewed regions and masks matter: the `regions + mask` fit improves substantially over the initial quick fit. For scientific use, continue with residual inspection and the reviewed-analysis readiness scaffold in Example 4.

## What to try next

Example 4 moves from quicklook classification to an audit-first reviewed-analysis scaffold. The emphasis changes from “can I get a plausible first fit?” to “what do I need to check so I get a good fit?”.


In [ ]:
sp.describe_public_function("fit_stellar_spectrum")
